# 09 — omniASR zero-shot di dev resmi (evaluasi gap, lead #2)

**Tujuan.** Satu angka yang menentukan apakah lead #2 (ganti model) layak dikejar
sebelum 2 Okt: **berapa WER omniASR tanpa fine-tune di 372 klip dev**, dibandingkan
dengan D (E4-native beam5), yaitu dev 0,2268 dan test 0,2901.

Kenapa omniASR: halaman dataset dev di MDC menyebut referensinya dibuat oleh
**omniASR_LLM_3B lalu diedit manual**. Evaluasi gap #1 (`reports/gap-eval-1-reference-convention-2026-09-27.md`)
menemukan ~66 % error ejaan varian kita berasal dari konvensi data latih. Model
pembuat referensi mestinya mendapatkan konvensi itu tanpa usaha. Lisensi bobot dan
kodenya Apache 2.0, jadi lolos R6 dan R14.

| # | Pertanyaan | Kenapa penting |
|---|---|---|
| Q1 | WER LLM_3B (generator referensi) di dev, dengan token `jav_Latn` / tanpa / `ind_Latn` | Kalau jauh di bawah 0,2268, gap ke #1 kemungkinan besar ada di sini |
| Q2 | v1 vs v2, 3B vs 1B, LLM vs CTC | CTC bisa dikonversi ke `Wav2Vec2ForCTC` (transformers 4.x), jadi jalur runtime-nya paling realistis |
| Q3 | Seberapa besar keunggulannya konvensi saja (lowercase + tanpa tanda baca) | Memisahkan efek "ejaan referensi" dari "pendengaran" |
| Q4 | Kecepatan dan VRAM | Test 2.118 klip harus selesai < 2 jam di A100 |

**Bukan** pertanyaan di sini: cara memasukkan model ini ke container. Paket
`omnilingual-asr` gagal di runtime (fairseq2 mengunci torch 2.8 / numpy<2, runtime
memakai torch 2.11). Itu baru dipikirkan kalau angkanya layak.

**Cara menjalankan (Kaggle):**
1. Accelerator **GPU T4** (bukan P100: wheel torch 2.8 cu128 tidak mendukung sm_60).
2. **Internet ON** (pip + unduh bobot dari `dl.fbaipublicfiles.com`).
3. Attach dataset **`indonesian_dev`**. Tidak perlu HF_TOKEN.
4. Jalankan dulu dengan `RUN_MODE="smoke"` (~20–30 menit, sebagian besar unduhan),
   lalu `RUN_MODE="full"`.
5. Salin blok `CLAUDE-HANDOFF-JSON` ke chat. Isinya **hanya agregat**. Download
   `findings/09-omniasr-zeroshot*/` ke repo seperti biasa. File prediksinya dianalisis
   lokal oleh skrip yang hanya mencetak agregat.

Paket omniASR dipasang di **venv terpisah** (`/tmp/omni-venv`) dan dijalankan sebagai
subprocess. Pin torch 2.8 dan numpy<2 tidak menyentuh environment Kaggle yang dipakai
notebook ini untuk scoring.

In [ ]:
# ============================================================
# CELL 1 - KNOBS
# ============================================================
RUN_MODE   = "full"         # smoke lolos 28 Sep (12/12, fp16 aman); sekarang full
SMOKE_N    = 12
DTYPE      = "auto"         # auto = bf16 di GPU sm>=80, fp16 di T4. Smoke membandingkan
                            # fp16 vs bf16 di klip yang sama; kalau fp16 rusak, set "bf16".
BATCH_SIZE = 4              # klip per panggilan transcribe(); klip diurutkan per durasi
TIME_BUDGET_H = 3.0         # full: config yang proyeksinya melewati budget dilewati, bukan dipotong
NOTEBOOK_ID = "09-omniasr-zeroshot"

# Urut prioritas. Dikelompokkan per model card: bobot diunduh sekali, lalu dihapus
# sebelum card berikutnya (disk Kaggle terbatas; LLM 3B = 17 GiB).
GRID_FULL = [
    # Dipersempit 28 Sep setelah smoke: di 12 klip LLM_3B zero-shot (0,2297) tidak lebih
    # baik dari D (0,2178) di klip yang sama. Full hanya perlu menjawab di n=372: apakah v1/v2
    # mendengar lebih baik dari D (lower_nopunct), dan seberapa jauh jalur CTC tertinggal.
    ("omniASR_LLM_3B",    "jav_Latn"),   # generator referensi dev (menurut halaman MDC)
    ("omniASR_LLM_3B_v2", "jav_Latn"),
    ("omniASR_CTC_3B_v2", None),         # satu-satunya jalur yang realistis masuk container
    # Tidak dijalankan (nilai keputusan kecil dibanding ~40 menit T4 per config):
    # ("omniASR_LLM_3B", None), ("omniASR_LLM_3B", "ind_Latn"), ("omniASR_LLM_3B_v2", None),
    # ("omniASR_LLM_1B_v2", "jav_Latn"), ("omniASR_CTC_1B_v2", None),
]
CARD_GIB = {"omniASR_LLM_3B": 17.0, "omniASR_LLM_3B_v2": 17.0, "omniASR_CTC_3B_v2": 12.0,
            "omniASR_LLM_1B_v2": 8.5, "omniASR_CTC_1B_v2": 3.7}

# Pembanding, dihitung di laptop dari prediksi dev tersimpan dengan normalizer resmi.
# lower_nopunct = lowercase + semua tanda baca (termasuk elipsis dan tanda hubung) dibuang.
KNOWN = {
    "D (E4-native beam5)": dict(wer=0.2268, clean222=0.2265, lower_nopunct=0.2115, test=0.2901),
    "B (E4 beam5)":        dict(wer=0.2260, clean222=0.2291, lower_nopunct=0.2108, test=0.2950),
}

## Setup

Discovery dataset berbasis isi (bukan path tetap), sama seperti NB 06/08. Keluaran
ke `/kaggle/working/findings/<RUN_ID>/`.

In [ ]:
# ============================================================
# CELL 2 - DISCOVERY, OUTPUT, HELPERS
# ============================================================
import os, sys, re, json, time, shutil, subprocess, collections, hashlib
from pathlib import Path
import numpy as np, pandas as pd

def head(t):
    print("\n" + "=" * 74); print(t); print("=" * 74)

assert RUN_MODE in ("smoke", "full"), RUN_MODE
head("DISCOVERY")
RUN_ID  = NOTEBOOK_ID + ("-smoke" if RUN_MODE == "smoke" else "")
OUT_DIR = Path("/kaggle/working/findings") / RUN_ID
PRED_DIR = OUT_DIR / "pred"
PRED_DIR.mkdir(parents=True, exist_ok=True)

INPUT = Path("/kaggle/input")
_hits = sorted(INPUT.glob("**/indonesian_dev/metadata.tsv")) or \
        sorted(p for p in INPUT.glob("**/metadata.tsv") if (p.parent / "clips").is_dir())
if not _hits:
    raise FileNotFoundError("dev set tidak ditemukan - attach dataset indonesian_dev")
DEV_META  = _hits[0]
DEV_CLIPS = DEV_META.parent / "clips"
DEV = pd.read_csv(DEV_META, sep="\t", dtype=str, keep_default_na=False)
assert len(DEV) == 372, "dev resmi seharusnya 372 klip, dapat %d" % len(DEV)
print("dev          :", DEV_META, "|", len(DEV), "baris |", len(list(DEV_CLIPS.glob("*.mp3"))), "mp3")
# Subset seleksi checkpoint NB 06, direkonstruksi persis seperti di NB 08.
SEL150 = set(DEV.sample(n=150, random_state=20260829)["audio_filename"])

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("gpu          :", gpu or "TIDAK ADA")
if not gpu:
    raise RuntimeError("Tidak ada GPU. Settings -> Accelerator -> GPU T4.")
try:
    cap = float(gpu.split(",")[-1].strip())
except ValueError:
    cap = 7.5 if "T4" in gpu else (6.0 if "P100" in gpu else 8.0)
if cap < 7.0:
    raise RuntimeError("GPU compute capability %.1f: torch 2.8 cu128 butuh >= 7.0. Pilih T4." % cap)
RESOLVED_DTYPE = DTYPE if DTYPE != "auto" else ("bf16" if cap >= 8.0 else "fp16")
print("dtype        :", RESOLVED_DTYPE, "(knob: %s)" % DTYPE)
du = shutil.disk_usage("/tmp")
print("disk /tmp    : %.1f GiB bebas" % (du.free / 2**30))

FINDINGS = {}
def rec(section, **kw):
    FINDINGS[section] = {**FINDINGS.get(section, {}), **kw}

def dump_handoff():
    blob = json.dumps(FINDINGS, ensure_ascii=False, separators=(",", ":"), sort_keys=True, default=str)
    print("\n===== BEGIN CLAUDE-HANDOFF-JSON =====")
    print(blob)
    print("===== END CLAUDE-HANDOFF-JSON =====")
    print("size: %.1f KB" % (len(blob) / 1024))

rec("env", run_id=RUN_ID, run_mode=RUN_MODE, gpu=gpu, dtype=RESOLVED_DTYPE, batch_size=BATCH_SIZE,
    disk_free_gib=round(du.free / 2**30, 1), n_dev=len(DEV))

## venv omniASR

`uv` membuat venv dari Python sistem, lalu memasang `omnilingual-asr` dengan semua
pin-nya (fairseq2 ≤ 0.6, torch 2.8, numpy < 2) di dalamnya. Cek terakhir menjalankan
impor di venv itu dan membaca versi torch serta ketersediaan CUDA.

In [ ]:
# ============================================================
# CELL 3 - VENV omnilingual-asr (terisolasi dari kernel)
# ============================================================
head("VENV")
VENV = Path("/tmp/omni-venv")
VPY  = VENV / "bin" / "python"
def sh(cmd, check=True):
    t0 = time.time()
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-6:]
    print("$ %s  [%.0f s, exit %d]" % (cmd[:110], time.time() - t0, r.returncode))
    for line in tail: print("   ", line[:200])
    if check and r.returncode != 0:
        full = (r.stdout + r.stderr).strip().splitlines()[-20:]
        raise RuntimeError("perintah gagal: " + cmd[:120] + "\n" + "\n".join(l[:220] for l in full))
    return r

t_venv = time.time()
# fairseq2n menaut ke libsndfile sistem.
sh("ldconfig -p | grep -q libsndfile || (apt-get -qq update && apt-get -qq install -y libsndfile1)", check=False)
if not VPY.exists():
    sh("pip install -q uv")
    sh("uv venv %s --python %s --seed -q" % (VENV, sys.executable))
# Selalu dijalankan (uv melewati yang sudah terpasang). torchaudio WAJIB sama versi dengan
# torch: tanpa pin, uv mengambil torchaudio 2.11 (CUDA 13) di samping torch 2.8 dari
# fairseq2n, dan impornya gagal mencari libcudart.so.13 (run 28 Sep). wrapt juga tidak
# ikut terpasang otomatis.
sh("uv pip install --python %s -q omnilingual-asr torch==2.8.0 torchaudio==2.8.0 wrapt librosa soundfile" % VPY)
# Diagnosa per impor. Kalau ada yang gagal, pesan error-nya DIMASUKKAN ke exception,
# supaya ikut ter-copy bersama traceback (output di atas traceback sering terlewat).
diag = []
for mod in ("torch", "numpy", "fairseq2n", "fairseq2", "omnilingual_asr"):
    r = subprocess.run([str(VPY), "-c", "import %s" % mod], capture_output=True, text=True)
    diag.append("import %-16s %s" % (mod, "OK" if r.returncode == 0 else "GAGAL"))
    if r.returncode != 0:
        diag += ["    " + l[:220] for l in r.stderr.strip().splitlines()[-25:]]
pkgs = subprocess.run("%s -m pip list 2>/dev/null | grep -iE '^(torch|torchaudio|fairseq2|fairseq2n|numpy|omnilingual|kenlm) '"
                      % VPY, shell=True, capture_output=True, text=True).stdout.strip()
diag += ["paket: " + " | ".join(" ".join(l.split()) for l in pkgs.splitlines())]
print("\n".join(diag))
if any(l.endswith("GAGAL") for l in diag):
    raise RuntimeError("impor di venv gagal:\n" + "\n".join(diag))
probe = sh("%s -c \"import torch, numpy, omnilingual_asr, fairseq2; "
           "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), "
           "'| numpy', numpy.__version__, '| fairseq2', getattr(fairseq2, '__version__', '?'), "
           "'| omnilingual_asr', getattr(omnilingual_asr, '__version__', '?'))\"" % VPY)
probe_line = [l for l in probe.stdout.splitlines() if l.startswith("torch")][-1]
print("venv         :", probe_line)
assert "cuda True" in probe_line, "torch di venv tidak melihat CUDA"
langs = sh("%s -c \"from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs as s; "
           "print(len(s), 'jav_Latn' in s, 'ind_Latn' in s)\"" % VPY).stdout.split()
print("bahasa       : %s didukung | jav_Latn %s | ind_Latn %s" % tuple(langs[-3:]))
assert langs[-2:] == ["True", "True"], "jav_Latn / ind_Latn tidak ada di supported_langs"
rec("venv", probe=probe_line, n_langs=int(langs[-3]), setup_s=round(time.time() - t_venv))

## Worker

Satu subprocess per config (card, bahasa, dtype). Audio dibaca dengan **loader yang
sama dengan `main.py`** (librosa, 16 kHz, mono), lalu dikirim sebagai dict
`{"waveform", "sample_rate"}`. Klip diurutkan per durasi supaya padding per batch kecil,
dan urutannya dikembalikan sebelum ditulis. Batch yang gagal diulang per klip. Klip
yang tetap gagal ditulis kosong dan dihitung. Batas 40 s dari pipeline dijaga dengan
memotong di 39,9 s dan menghitung berapa klip yang terpotong (dev maksimum 37,9 s,
jadi seharusnya 0).

In [ ]:
# ============================================================
# CELL 4 - WORKER SCRIPT
# ============================================================
WORKER = Path("/tmp/omni_worker.py")
WORKER.write_text(r"""
import sys, os, json, time, csv, traceback
import numpy as np, torch, librosa
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline

job = json.load(open(sys.argv[1]))
SR = 16000
dtype = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}[job["dtype"]]
t0 = time.time()
pipe = ASRInferencePipeline(model_card=job["card"], dtype=dtype)
init_s = time.time() - t0

names = job["names"]
audio, n_trunc = [], 0
for n in names:
    y, _ = librosa.load(os.path.join(job["clips"], n), sr=SR, mono=True)
    if len(y) > int(39.9 * SR):
        y = y[: int(39.9 * SR)]; n_trunc += 1
    audio.append(y.astype(np.float32))
audio_s = sum(len(y) for y in audio) / SR
order = sorted(range(len(audio)), key=lambda i: len(audio[i]))
lang = job["lang"]
B = job["batch_size"]
hyp = [""] * len(audio)
n_fail, errs = 0, []

def run(idx):
    batch = [{"waveform": audio[i], "sample_rate": SR} for i in idx]
    langs = [lang] * len(idx) if lang else None
    return pipe.transcribe(batch, lang=langs, batch_size=len(idx))

torch.cuda.reset_peak_memory_stats()
t1 = time.time()
for k in range(0, len(order), B):
    idx = order[k:k + B]
    try:
        out = run(idx)
        for i, t in zip(idx, out): hyp[i] = t or ""
    except Exception as e:
        errs.append(type(e).__name__ + ": " + str(e)[:160])
        for i in idx:
            try:
                hyp[i] = run([i])[0] or ""
            except Exception as e2:
                n_fail += 1; errs.append(type(e2).__name__ + ": " + str(e2)[:160])
    if (k // B) % 20 == 0:
        print("  progress %d/%d clips, %.0f s" % (min(k + B, len(order)), len(order), time.time() - t1), flush=True)
torch.cuda.synchronize()
decode_s = time.time() - t1

with open(job["out_csv"], "w", newline="", encoding="utf-8") as fh:
    w = csv.writer(fh); w.writerow(["audio_filename", "transcript"])
    for n, t in zip(names, hyp): w.writerow([n, " ".join(t.split())])
json.dump(dict(init_s=round(init_s, 1), decode_s=round(decode_s, 1), audio_s=round(audio_s, 1),
               rtf=round(decode_s / max(audio_s, 1e-9), 4), n=len(names), n_fail=n_fail, n_trunc=n_trunc,
               peak_vram_gib=round(torch.cuda.max_memory_allocated() / 2**30, 2),
               torch=torch.__version__, errors=errs[:5]),
          open(job["out_meta"], "w"))
print("  done: init %.0f s, decode %.0f s, RTF %.3f, fail %d" % (init_s, decode_s, decode_s / max(audio_s, 1e-9), n_fail), flush=True)
""")
print("worker ditulis:", WORKER, "|", len(WORKER.read_text().splitlines()), "baris")

## Scoring

`normalize_text` disalin apa adanya dari `score.py` resmi, lewat NB 08 sel 8. Selain
WER resmi, dicatat juga **`lower_nopunct`** (lowercase + semua tanda baca dibuang, di
kedua sisi) untuk Q3. Selisih antara keduanya adalah beban konvensi kosmetik.

In [ ]:
# ============================================================
# CELL 5 - NORMALIZER RESMI + METRIK AGREGAT
# ============================================================
def _lowercase_sentence_initial(m):
    delimiter, first, second = m.group(1), m.group(2), m.group(3)
    if first.isupper() and not (second and second.isupper()): first = first.lower()
    return delimiter + first + second

def normalize_text(text):
    BRACKETED = re.compile(r"\[[^\]]+\]"); UNINTELLIGIBLE_PAREN = re.compile(r"\(\?+\)")
    WORD_PAREN = re.compile(r"\(([^()]*)\)"); PUNCTUATION_OTHER = re.compile('[¿¡";:]+')
    COMMA = re.compile(",+"); SENTENCE_INITIAL = re.compile(r"(^\s*|[.!?—]\s*)([^\W\d_])([^\W\d_]?)")
    SENTENCE_END = re.compile("[!?]+"); MULTISPACE = re.compile("  +")
    text = text.replace("~", "")
    text = re.sub(BRACKETED, " ", text); text = re.sub(UNINTELLIGIBLE_PAREN, " ", text)
    text = re.sub(WORD_PAREN, r"\1", text); text = text.replace("#x27;", "'")
    text = re.sub(PUNCTUATION_OTHER, " ", text)
    text = re.sub(SENTENCE_INITIAL, _lowercase_sentence_initial, text)
    text = text.replace("—", ", "); text = re.sub(COMMA, " ", text)
    text = re.sub(SENTENCE_END, " ", text)
    text = text.replace("...", "!ELLIPSIS!").replace(".", " ").replace("!ELLIPSIS!", "...")
    while " ... " in text: text = text.replace(" ... ", " ")
    return re.sub(MULTISPACE, " ", text)

_PUN = re.compile(r"[^\w]+")
def _lower_nopunct(tokens):
    return [t for t in (_PUN.sub("", x.lower()) for x in tokens) if t]

def _sdi(r, h):
    # Levenshtein dengan backtrace; mengembalikan (S, D, I).
    n, m = len(r), len(h)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1): D[i][0] = i
    for j in range(m + 1): D[0][j] = j
    for i in range(1, n + 1):
        ri = r[i - 1]; Di = D[i]; Dp = D[i - 1]
        for j in range(1, m + 1):
            Di[j] = min(Dp[j - 1] + (ri != h[j - 1]), Dp[j] + 1, Di[j - 1] + 1)
    s = d = ins = 0; i, j = n, m
    while i or j:
        if i and j and D[i][j] == D[i - 1][j - 1] + (r[i - 1] != h[j - 1]):
            s += r[i - 1] != h[j - 1]; i -= 1; j -= 1
        elif i and D[i][j] == D[i - 1][j] + 1:
            d += 1; i -= 1
        else:
            ins += 1; j -= 1
    return s, d, ins

def score_config(pred_csv):
    P = pd.read_csv(pred_csv, dtype=str, keep_default_na=False)
    hyp = dict(zip(P["audio_filename"], P["transcript"]))
    rows = DEV[DEV["audio_filename"].isin(hyp)]
    agg = collections.Counter(); by_lang = collections.Counter()
    for _, r in rows.iterrows():
        rt, ht = normalize_text(r["transcript"]).split(), normalize_text(hyp[r["audio_filename"]]).split()
        s, d, i = _sdi(rt, ht); e = s + d + i
        agg["S"] += s; agg["D"] += d; agg["I"] += i; agg["N"] += len(rt); agg["H"] += len(ht)
        rl, hl = _lower_nopunct(rt), _lower_nopunct(ht)
        agg["E_lnp"] += sum(_sdi(rl, hl)); agg["N_lnp"] += len(rl)
        if r["audio_filename"] not in SEL150:
            agg["E_c"] += e; agg["N_c"] += len(rt)
        by_lang[r["language"] + "_E"] += e; by_lang[r["language"] + "_N"] += len(rt)
        agg["empty"] += not ht
        agg["digit_tok"] += sum(any(c.isdigit() for c in t) for t in ht)
    E = agg["S"] + agg["D"] + agg["I"]
    out = dict(n_clips=len(rows), n_err=E, n_ref=agg["N"], wer=round(E / max(agg["N"], 1), 4),
               S=agg["S"], D=agg["D"], I=agg["I"],
               wer_lower_nopunct=round(agg["E_lnp"] / max(agg["N_lnp"], 1), 4),
               hyp_ref_len_ratio=round(agg["H"] / max(agg["N"], 1), 3),
               n_empty=agg["empty"], digit_tok_share=round(agg["digit_tok"] / max(agg["H"], 1), 4))
    if agg["N_c"] and RUN_MODE == "full":
        out["wer_clean222"] = round(agg["E_c"] / agg["N_c"], 4)
    for lg in sorted({k[:-2] for k in by_lang}):
        out["wer_" + lg] = round(by_lang[lg + "_E"] / max(by_lang[lg + "_N"], 1), 4)
    return out

# Self-test normalizer dengan kalimat sintetis (bukan data kompetisi).
_t = [("Aku mangan. Terus turu!", "aku mangan terus turu"),
      ("Ya... ngono, lho", "ya... ngono lho"),
      ("[noise] Budi (?) teka", "budi teka")]
_ok = all(" ".join(normalize_text(a).split()) == b for a, b in _t)
print("normalizer self-test:", "PASS" if _ok else "FAIL",
      [" ".join(normalize_text(a).split()) for a, _ in _t])
assert _sdi("a b c".split(), "a x c d".split()) == (1, 0, 1)
rec("scorer", self_test=_ok)

## Menjalankan grid

Smoke: 12 klip (sampel tetap). Isinya LLM_3B `jav_Latn` di dtype terpilih, LLM_3B
`jav_Latn` di bf16 (hanya kalau dtype terpilih bukan bf16, untuk mengecek fp16 tidak
rusak), dan CTC_1B_v2 (unduhan kecil, untuk menguji jalur CTC). Full: `GRID_FULL`
sesuai urutan prioritas, dengan budget waktu.

In [ ]:
# ============================================================
# CELL 6 - GRID
# ============================================================
head("GRID - " + RUN_MODE)
if RUN_MODE == "smoke":
    NAMES = list(DEV.sample(n=SMOKE_N, random_state=20260927)["audio_filename"])
    GRID = [("omniASR_LLM_3B", "jav_Latn", RESOLVED_DTYPE)]
    if RESOLVED_DTYPE != "bf16":
        GRID.append(("omniASR_LLM_3B", "jav_Latn", "bf16"))
    GRID.append(("omniASR_CTC_1B_v2", None, RESOLVED_DTYPE))
else:
    NAMES = list(DEV["audio_filename"])
    GRID = [(c, l, RESOLVED_DTYPE) for c, l in GRID_FULL]

CACHE = Path("/tmp/omni-cache")
ENV = dict(os.environ, XDG_CACHE_HOME=str(CACHE), FAIRSEQ2_CACHE_DIR=str(CACHE / "fairseq2"),
           CUDA_VISIBLE_DEVICES="0", PYTHONUNBUFFERED="1")
dev_audio_s = 24.9 * len(NAMES)   # rata-rata durasi dev (analisis 28 Agu), untuk proyeksi saja
RTF_GUESS = {"LLM": 0.35, "CTC": 0.03}
rtf_seen = {}
T_START = time.time(); RESULTS = {}; SKIPPED = {}
current_card = None

def tag(card, lang, dt):
    return "%s__%s__%s" % (card, lang or "nolang", dt)

for card, lang, dt in GRID:
    name = tag(card, lang, dt)
    fam = "CTC" if "_CTC_" in card else "LLM"
    if card != current_card and CACHE.exists():
        shutil.rmtree(CACHE, ignore_errors=True)        # bobot card sebelumnya
    free = shutil.disk_usage("/tmp").free / 2**30
    if card != current_card and free < CARD_GIB.get(card, 17) + 4:
        SKIPPED[name] = "disk %.1f GiB bebas" % free; print("SKIP", name, SKIPPED[name]); continue
    proj_h = (dev_audio_s * rtf_seen.get((card, dt), RTF_GUESS[fam]) + (0 if card == current_card else 600)) / 3600
    used_h = (time.time() - T_START) / 3600
    if RUN_MODE == "full" and used_h + proj_h > TIME_BUDGET_H:
        SKIPPED[name] = "budget: terpakai %.2f h + proyeksi %.2f h > %.1f h" % (used_h, proj_h, TIME_BUDGET_H)
        print("SKIP", name, SKIPPED[name]); continue
    print("\n>>> %s  (proyeksi %.2f h, terpakai %.2f h)" % (name, proj_h, used_h))
    job = dict(card=card, lang=lang, dtype=dt, names=NAMES, clips=str(DEV_CLIPS), batch_size=BATCH_SIZE,
               out_csv=str(PRED_DIR / (name + ".csv")), out_meta=str(PRED_DIR / (name + ".meta.json")))
    jp = Path("/tmp/job_%s.json" % name); jp.write_text(json.dumps(job))
    t0 = time.time()
    r = subprocess.run([str(VPY), str(WORKER), str(jp)], env=ENV, capture_output=True, text=True)
    lines = [l for l in (r.stdout + r.stderr).splitlines() if l.strip()]
    for l in [l for l in lines if l.startswith("  ")][-4:]: print(l[:200])
    current_card = card
    if r.returncode != 0 or not Path(job["out_meta"]).exists():
        SKIPPED[name] = "worker exit %d: %s" % (r.returncode, (lines[-1] if lines else "")[:200])
        print("GAGAL", name); print("\n".join(l[:200] for l in lines[-15:])); continue
    meta = json.loads(Path(job["out_meta"]).read_text())
    rtf_seen[(card, dt)] = meta["rtf"]
    res = score_config(job["out_csv"]); res.update(meta); res["wall_s"] = round(time.time() - t0)
    RESULTS[name] = res
    print("    WER %.4f | lower_nopunct %.4f | S/D/I %d/%d/%d | kosong %d | gagal %d | RTF %.3f | VRAM %.1f GiB"
          % (res["wer"], res["wer_lower_nopunct"], res["S"], res["D"], res["I"], res["n_empty"],
             res["n_fail"], res["rtf"], res["peak_vram_gib"]))
    rec("results", **{name: res})

shutil.rmtree(CACHE, ignore_errors=True)
rec("grid", n_planned=len(GRID), n_done=len(RESULTS), skipped=SKIPPED,
    total_h=round((time.time() - T_START) / 3600, 2))

## Ringkasan

Dibandingkan dengan D. Di smoke, WER 12 klip **tidak bermakna** sebagai estimasi. Yang
dilihat: jalurnya jalan, fp16 tidak rusak dibanding bf16, kecepatan, dan VRAM.

In [ ]:
# ============================================================
# CELL 7 - TABEL + PROYEKSI RUNTIME
# ============================================================
head("RINGKASAN")
if RESULTS:
    cols = ["wer", "wer_lower_nopunct", "wer_clean222", "wer_javind", "wer_ind", "S", "D", "I",
            "hyp_ref_len_ratio", "n_empty", "digit_tok_share", "rtf", "peak_vram_gib", "init_s"]
    T = pd.DataFrame(RESULTS).T.reindex(columns=cols)
    print(T.to_string())
    T.to_csv(OUT_DIR / "results.csv")
print("\npembanding (dev 372):")
for k, v in KNOWN.items(): print("  %-22s %s" % (k, v))
if RUN_MODE == "full" and RESULTS:
    best = min(RESULTS, key=lambda k: RESULTS[k]["wer"])
    b = RESULTS[best]; d = KNOWN["D (E4-native beam5)"]
    print("\nterbaik: %s  WER %.4f (D %.4f, selisih %+.4f) | lower_nopunct %.4f (D %.4f, %+.4f)"
          % (best, b["wer"], d["wer"], b["wer"] - d["wer"], b["wer_lower_nopunct"], d["lower_nopunct"],
             b["wer_lower_nopunct"] - d["lower_nopunct"]))
    rec("best", config=best, wer=b["wer"], delta_vs_D=round(b["wer"] - d["wer"], 4),
        lnp_delta_vs_D=round(b["wer_lower_nopunct"] - d["lower_nopunct"], 4))
# Proyeksi kasar test: 2.118 klip x ~24,9 s. A100 kira-kira 2-3x lebih cepat dari T4
# (belum diukur); angka T4 dicatat apa adanya.
for k, v in RESULTS.items():
    h = 2118 * 24.9 * v["rtf"] / 3600
    print("  proyeksi test di GPU ini: %-45s %.2f h" % (k, h))
    rec("projection", **{k: round(h, 2)})

## Self-validation

In [ ]:
# ============================================================
# CELL 8 - SELF-VALIDATION
# ============================================================
head("SELF-VALIDATION")
CHECKS = []
def check(name, cond, detail=""):
    CHECKS.append(dict(check=name, passed=bool(cond), detail=str(detail)[:160]))
    print("  [%s] %-60s %s" % ("PASS" if cond else "FAIL", name, detail))

check("normalizer resmi lolos self-test", FINDINGS.get("scorer", {}).get("self_test"))
check("venv: torch melihat CUDA", "cuda True" in FINDINGS.get("venv", {}).get("probe", ""))
check("config pertama (LLM_3B jav_Latn) selesai", tag(*GRID[0]) in RESULTS, tag(*GRID[0]))
check("semua config selesai atau dilewati dengan alasan",
      len(RESULTS) + len(SKIPPED) == len(GRID), "%d selesai, %d dilewati" % (len(RESULTS), len(SKIPPED)))
for k, v in RESULTS.items():
    check("%s: semua klip ada, 0 gagal" % k[:40], v["n_clips"] == len(NAMES) and v["n_fail"] == 0,
          "%d/%d, gagal %d" % (v["n_clips"], len(NAMES), v["n_fail"]))
    check("%s: kosong <= 2%%, tidak ada klip terpotong" % k[:40],
          v["n_empty"] <= 0.02 * len(NAMES) and v["n_trunc"] == 0, "kosong %d, potong %d" % (v["n_empty"], v["n_trunc"]))
if RUN_MODE == "smoke" and RESOLVED_DTYPE != "bf16":
    a, b = tag("omniASR_LLM_3B", "jav_Latn", RESOLVED_DTYPE), tag("omniASR_LLM_3B", "jav_Latn", "bf16")
    if a in RESULTS and b in RESULTS:
        gap = RESULTS[a]["wer"] - RESULTS[b]["wer"]
        check("fp16 tidak rusak (WER fp16 - bf16 <= 0,02 di 12 klip)", gap <= 0.02, "%+.4f" % gap)
        rec("dtype_check", fp16_minus_bf16=round(gap, 4),
            speed_ratio_bf16_over_fp16=round(RESULTS[b]["rtf"] / max(RESULTS[a]["rtf"], 1e-9), 2))
# Handoff hanya boleh berisi agregat: tidak ada transkrip hipotesis >= 4 kata di dalamnya.
blob = json.dumps(FINDINGS, ensure_ascii=False, default=str)
leak = 0
for p in PRED_DIR.glob("*.csv"):
    for h in pd.read_csv(p, dtype=str, keep_default_na=False)["transcript"]:
        leak += len(h.split()) >= 4 and h in blob
check("handoff tanpa transkrip", leak == 0, "%d kebocoran" % leak)
rec("checks", passed=sum(c["passed"] for c in CHECKS), total=len(CHECKS),
    failed=[c["check"] for c in CHECKS if not c["passed"]])
print("\n%d/%d PASS" % (sum(c["passed"] for c in CHECKS), len(CHECKS)))

## Handoff

Salin seluruh blok di antara penanda `BEGIN` / `END` ke chat. Isinya agregat saja.
Lalu Output → Download all → unzip ke root repo (`findings/<RUN_ID>/`).

In [ ]:
# ============================================================
# CELL 9 - DIGEST
# ============================================================
head("FINGERPRINTS")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        rec("files", **{str(p.relative_to(OUT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest()[:16]})
(OUT_DIR / "dataset_findings.json").write_text(json.dumps(FINDINGS, indent=1, sort_keys=True, default=str),
                                               encoding="utf-8")
print("ditulis:", OUT_DIR / "dataset_findings.json")
dump_handoff()